# CSA — danh từ/động từ TRAIN + ROI thật (CPU)

**Accelerator None, Internet ON**. Add Input `visual-cache`, `objectdetectionecache`,
`mscoco-2014`. Không cần checkpoint/prompt/UPT/token. Cell đầu có4path nếu mount khác.

1. Cài spaCy3.8.11 trong venv tạm, kiểm SHA model en_core_web_sm3.8.0.
2. Pool raw CLIP patches theo box thật cho256TRAIN+64val; không dùng test.
3. POS toàn566747captionTRAIN để đếm noun/verb lemma, giữ tần suất≥5.

Có chạy model POS nhỏ trên CPU; **chưa chạy CLIP text/captioner hoặc huấn luyện**.
Worker cap2giờ là giới hạn dừng, không ETA. Không bật GPU cho giai đoạn này.
Save & Run All. Gửi `csa_paper_real_preparation_review.zip`.
**Giữ toàn bộ folder `csa_paper_real_preparation` để lưu thành Dataset**;
`regions.npz` không nằm trong ZIP review, không xóa nó. Bước sau mới mã hóa
các từ bằng CLIP text và kiểm CSA với đặc trưng thật.


In [ ]:
import hashlib, json, os, queue, signal, subprocess, sys, tempfile, threading, time, traceback, urllib.request, zipfile
from pathlib import Path
COMMIT = '9b7567ad31cb3709b922239d7a17d29140a93d26'
HASHES = {'prepare_real_assets.py': '44a22d92acd239bdc7a1edbe30984daf0853ac8bc196969dd48b19b78df1cc7b', 'audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'test_real_assets.py': '0a18eb3b8c5bf519e1c0c7a898760216e4e57ecadc182947640b50eb53bb5c04', 'REAL_ASSET_PREPARATION_PROTOCOL.md': '668c751157f550b4d65d9824ac09a4c6098ea8a092b00349283bbe3c41557fec'}
OUTPUT = Path('/kaggle/working/csa_paper_real_preparation')
COCO_JSON = Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS = Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
VISUAL_CACHE = Path('/kaggle/input/datasets/ducanh2403/visual-cache')
IMAGE_ROOT = COCO_JSON.parent/'images'
if OUTPUT.exists(): raise RuntimeError('Preserve previous output and use a fresh session.')
OUTPUT.mkdir(parents=True)
SOURCE = Path(tempfile.mkdtemp(prefix='csa-real-prepare-'))
LOG = OUTPUT/'notebook.log'
CONFIG = SOURCE/'config.json'
CONFIG.write_text(json.dumps(dict(dataset_json=str(COCO_JSON), detections=str(DETECTIONS), visual_cache=str(VISUAL_CACHE), image_root=str(IMAGE_ROOT), output=str(OUTPUT))), encoding='utf-8')
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT, sha256=HASHES, pos_model_url='https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl', pos_model_sha256='1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85'), indent=2), encoding='utf-8')
for name, expected in HASHES.items():
    payload = urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{name}', timeout=90).read()
    if hashlib.sha256(payload).hexdigest() != expected: raise RuntimeError('Source differs: '+name)
    (SOURCE/name).write_bytes(payload)
print('CPU preparation source verified. Accelerator must be None.', flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Smoke active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete = False
try:
    import numpy as np
    os.environ['OMP_NUM_THREADS'] = '2'
    os.environ['OPENBLAS_NUM_THREADS'] = '2'
    os.environ['CUDA_VISIBLE_DEVICES'] = ''
    venv = SOURCE/'pos-env'
    run([sys.executable, '-m', 'venv', '--system-site-packages', venv], timeout=180)
    PYTHON = venv/'bin/python'
    constraint = SOURCE/'constraints.txt'
    constraint.write_text('numpy=='+np.__version__+chr(10), encoding='utf-8')
    run([PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check', '--only-binary=:all:', '-c', constraint, 'spacy==3.8.11'], timeout=900)
    wheel = SOURCE/'en_core_web_sm-3.8.0-py3-none-any.whl'
    payload = urllib.request.urlopen('https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl', timeout=180).read()
    if hashlib.sha256(payload).hexdigest() != '1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85': raise RuntimeError('Official POS model wheel SHA differs')
    wheel.write_bytes(payload)
    run([PYTHON, '-m', 'pip', 'install', '--no-deps', '--force-reinstall', wheel], timeout=180)
    (OUTPUT/'software_freeze.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze', '--all'], text=True), encoding='utf-8')
    run([PYTHON, '-u', SOURCE/'test_real_assets.py'], timeout=120)
    run([PYTHON, '-u', SOURCE/'prepare_real_assets.py', '--config', CONFIG], timeout=7200)
    summary = json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))
    receipt = json.loads((OUTPUT/'receipt.json').read_text(encoding='utf-8'))
    if summary.get('status') != 'CPU_PREPARATION_COMPLETE' or receipt.get('status') != 'CPU_PREPARATION_COMPLETE': raise RuntimeError('Preparation incomplete')
    for name, expected in receipt['payload_sha256'].items():
        if Path(name).name != name or hashlib.sha256((OUTPUT/name).read_bytes()).hexdigest() != expected: raise RuntimeError('Output payload differs: '+name)
    complete = True
    print(json.dumps(summary, indent=2), flush=True)
except BaseException:
    (OUTPUT/'notebook_failure.txt').write_text(traceback.format_exc(), encoding='utf-8')
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete, caption_model_executed=False, gpu_used=False, semantic_embeddings_created=False, training_ready=False)), encoding='utf-8')
    archive = OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.iterdir()):
            if p.is_file() and p.suffix in ('.json', '.txt', '.log'): z.write(p, OUTPUT.name+'/'+p.name)
    print('SEND REVIEW ZIP:', archive, flush=True)
    if complete: print('KEEP ENTIRE OUTPUT FOLDER AS DATASET:', OUTPUT, flush=True)
    else: print('INCOMPLETE: preserve logs; do not use as prepared training data.', flush=True)
